# Stretch: more pandas, worked solutions

The answers to the stretch notebook. Try each one before looking.

In [ ]:
import os
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

plays = pd.read_csv("data/clean/plays.csv")
artists = pd.read_csv("data/clean/artists.csv")

print(plays.shape, artists.shape)

## Stretch 1: a pivot table in one line

In [ ]:
plays["year"] = plays["played_at"].str[:4]

table = pd.crosstab(plays["genre"], plays["year"])

assert table.loc["Jazz", "2024"] == 72
assert table.loc["Jazz", "2025"] == 82
table

In [ ]:
table["change"] = table["2025"] - table["2024"]

assert table.loc["Folk", "change"] == -34
assert table["change"].idxmin() == "Folk"
assert set(table[table["change"] > 0].index) == {"Afrobeat", "Jazz"}
assert table["2024"].sum() + table["2025"].sum() == len(plays)

os.makedirs("output", exist_ok=True)
table.to_csv("output/genre_by_year_pandas.csv")
print("wrote output/genre_by_year_pandas.csv")
table

Folk fell most, by 34. Only Afrobeat (up 14) and Jazz (up 10) grew. The same
table as session 4, and the last `assert` is the same check that the two
year columns add up to every play.

In session 4 this took a function to build the dictionary of dictionaries,
another to turn it into rows, and a `DictWriter`. Here it is `crosstab`, one
subtraction, and `to_csv`. Now you know what that one line is doing.

`table["change"].idxmin()` gives the **label** of the smallest value, rather
than the value itself: "which genre", not "how much".

`crosstab` only counts. Its bigger sibling, `pivot_table`, can add up or
average instead. Total minutes per genre per year:

In [ ]:
minutes_table = plays.pivot_table(index="genre", columns="year", values="minutes_played", aggfunc="sum")
minutes_table.round(1)

This shape, one row per thing and one column per year, is what spreadsheet
people make all day. It is also, very often, the shape a chart wants:
session 9.

## Stretch 2: the average of a 0/1 column is a rate

In [ ]:
per_artist = plays.groupby("artist_name").agg(
    plays=("play_id", "count"),
    skip_rate=("skipped", "mean"),
)
per_artist["skip_pct"] = (100 * per_artist["skip_rate"]).round(1)

assert per_artist.loc["Circuit Sparrow", "plays"] == 7
assert per_artist.loc["Circuit Sparrow", "skip_pct"] == 28.6

per_artist.sort_values("skip_pct", ascending=False).head(3)

In [ ]:
regulars = per_artist[per_artist["plays"] >= 30]

assert regulars["skip_pct"].idxmax() == "Mireille Basset"
assert regulars.loc["Mireille Basset", "skip_pct"] == 20.0

regulars.sort_values("skip_pct", ascending=False).head(3)

Filtering the groups after the `groupby` is `HAVING COUNT(*) >= 30`.

**Which list to believe?** Without the minimum, the most-skipped artist is
Circuit Sparrow at 28.6%: 2 skips out of 7 plays. One fewer skip and the
rate halves. With at least 30 plays, it is Mireille Basset, 8 out of 40, at
20.0%. The same conclusion as the session 4 stretch, which is reassuring:
two different methods, the same answer.

The rate trick is worth remembering. Any yes/no column stored as 0 and 1
turns into a percentage with `.mean()`, with no second count to divide by.

## Stretch 3: merging on a name

In [ ]:
clash = plays.merge(artists, on="artist_name")
print(clash.columns.tolist())

In [ ]:
merged = plays.merge(
    artists[["artist_name", "formed_year"]],
    on="artist_name",
    validate="many_to_one",
)

assert len(merged) == len(plays)
assert "formed_year" in merged.columns
assert "genre_x" not in merged.columns
print(len(plays), "->", len(merged), "rows")

Picking the columns before the merge avoids the clash entirely, and makes
the merge say what it is for: "bring me each play's formed year".

If you really do want both versions of a column, `suffixes=("_play",
"_artist")` gives them better names than `_x` and `_y`. Here they agree
anyway, so there is no point keeping both.

`validate="many_to_one"` passes because every artist name appears once in
`artists.csv`. That is worth checking whenever you join on a **name** rather
than an ID, because names are not guaranteed to be unique. Session 4's
stretch found three different artists with a track called *Paper Signal*.
Merging on track name would have fanned out, and `validate` would have said
so.

## Stretch 4: decades, and the plays that go missing

In [ ]:
merged["decade"] = merged["formed_year"] // 10 * 10

per_decade = merged.groupby("decade").size()

assert per_decade.loc[2010] == 1087
per_decade

In [ ]:
print(per_decade.sum(), "plays counted, out of", len(plays))

2,130, not 2,183. **53 plays have vanished, without an error.**

They belong to Rosewood Lane and Cassette Revival, the two artists with no
`formed_year`. Their year is `NaN`, so their decade is `NaN` too, because any
sum involving a missing number is missing. And `groupby` **drops missing
keys by default**. The rows were not filtered by you; they were quietly left
out of every group.

This is the same decision as the session 4 stretch: skip, pretend, or label.
`groupby` picked "skip" for you, and did not mention it. The fix is to ask
for the missing group explicitly:

In [ ]:
per_decade_all = merged.groupby("decade", dropna=False).size()

assert per_decade_all.sum() == len(plays)
assert per_decade_all.isna().sum() == 0               # the counts are all real numbers
assert per_decade_all[per_decade_all.index.isna()].iloc[0] == 53
per_decade_all

Now the `NaN` row is there with its 53 plays, and the total is right. The
last two lines check different things: the **counts** are never missing,
but one of the **labels** is.

Also note the decades are `2010.0`, not `2010`: a column with a `NaN` in it
is stored as decimals, as you saw in the homework.

**The habit:** after a `groupby`, check that the groups add up to the
total. It is the same habit as checking the row count after a merge, and it
catches the same kind of silent loss.

## Stretch 5: who is in one table and not the other?

In [ ]:
never = artists[~artists["artist_name"].isin(plays["artist_name"])]["artist_name"]

assert set(never) == {"Iron Fernway", "Gravel Choir"}
never

`~` turns every `True` into `False` and back, so `~...isin(...)` means "not
in". It is `NOT IN` from SQL.

In [ ]:
played_names = plays[["artist_name"]].drop_duplicates()

found = artists.merge(played_names, on="artist_name", how="left", indicator=True)

assert len(found) == len(artists)
assert found["_merge"].value_counts()["left_only"] == 2
found["_merge"].value_counts()

In [ ]:
found[found["_merge"] == "left_only"][["artist_name", "genre", "country"]]

`indicator=True` does a left join and labels every row with where it was
found. `"left_only"` is the `WHERE p.play_id IS NULL` from session 7, and
it has the advantage that you can see the matched ones too, all in one
table.

In [ ]:
fanned = artists.merge(plays[["artist_name"]], on="artist_name", how="left", indicator=True)
print(len(artists), "artists ->", len(fanned), "rows with the whole of plays")

With the whole of `plays` on the right, each artist meets every one of their
plays: 2,185 rows, the 2,183 plays plus the two unmatched artists. That is
fan-out again, in a merge whose only job was to answer yes or no. Reducing
the right side to one row per artist first (`drop_duplicates`) is the same
"aggregate first" fix as the session, in its smallest form.

## Stretch 6: the top artist in each genre

In [ ]:
pairs = plays.groupby(["genre", "artist_name"]).agg(
    plays=("play_id", "count"),
    minutes=("minutes_played", "sum"),
).reset_index()

top = pairs.sort_values("minutes", ascending=False).groupby("genre").head(1)

assert top.set_index("genre").loc["Electronic", "artist_name"] == "Glass Tram"
assert len(top) == 8
top.round(1)

`groupby("genre").head(1)` keeps the first row of each group **in the order
the rows are already in**. That is why the sort comes first: it decides
what "first" means.

In [ ]:
top_two = pairs.sort_values("minutes", ascending=False).groupby("genre").head(2)
top_two.sort_values(["genre", "minutes"], ascending=[True, False]).round(1)

In [ ]:
electronic = top_two[top_two["genre"] == "Electronic"]["minutes"]
gap = electronic.iloc[0] - electronic.iloc[1]

assert round(gap, 1) == 5.8
print(f"Electronic: Glass Tram wins by {gap:.1f} minutes out of about 1,170")

**Electronic is the genre where the winner is barely a winner.** Glass Tram
beats DJ Kompas by under six minutes, out of nearly 1,200 each: about one
play's worth. In every other genre the gap is at least 17 minutes, and
usually far more.

By plays it is Rock that becomes a coin toss:

In [ ]:
by_plays = pairs.sort_values("plays", ascending=False).groupby("genre").head(2)
rock = by_plays[by_plays["genre"] == "Rock"][["artist_name", "plays"]]

assert list(rock["plays"]) == [64, 63]
rock

The Quiet Wires 64, Static Meadow 63. One play.

The same lesson as the tie in the session 4 homework and the busiest day in
session 6: a ranking always produces a winner, whether or not there really
is one. `head(2)` instead of `head(1)` costs nothing and tells you which
kind you have.